## 05. Ekstrakcija proteinskih embeddinga (ESM-2)

### Uvoz biblioteka

In [1]:
import os
import time
import numpy as np
import pandas as pd
import torch
from transformers import EsmTokenizer, EsmModel
from tqdm.auto import tqdm

### Konfiguracija i putanje

In [2]:
DATA_DIR = "../data/processed"
FEATURES_DIR = "../data/features"

os.makedirs(FEATURES_DIR, exist_ok=True)

# 6 slojeva, 320-dim izlaz,  ~ 8M parametara
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
MAX_LENGTH = 1022 # 1024 - 2 (rezervisano za <cls> i <eos> tokene)
BATCH_SIZE = 8
CHECKPOINT_EVERY = 20 # broj batch-eva između čuvanja privremenog checkpointa

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Koristi se uredjaj: {DEVICE}")

Koristi se uredjaj: cpu


### Učitavanje podataka

In [3]:
dataset_path = os.path.join(DATA_DIR, "dataset_final.csv")
dataset = pd.read_csv(dataset_path)

### Ekstrakcija embeddinga

In [4]:
seq_df = dataset[["Entry", "Sequence"]].reset_index(drop=True)

print(f"Ukupan broj proteina za ekstrakciju embeddinga: {len(seq_df)}")
print(f"Raspon dužina sekvenci: min={seq_df['Sequence'].str.len().min()}, "
f"max={seq_df['Sequence'].str.len().max()}, "
f"prosjek={seq_df['Sequence'].str.len().mean():.1f}")

n_truncated = (seq_df["Sequence"].str.len() > MAX_LENGTH).sum()
print(f"Broj sekvenci koje će biti skraćene (dužina > {MAX_LENGTH}): {n_truncated} "
f"({100 * n_truncated / len(seq_df):.2f}%)")

Ukupan broj proteina za ekstrakciju embeddinga: 7043
Raspon dužina sekvenci: min=25, max=34350, prosjek=603.6
Broj sekvenci koje će biti skraćene (dužina > 1022): 815 (11.57%)


### Učitavanje ESM-2 modela i tokenizatora

In [5]:
print(f"Učitavanje modela: {MODEL_NAME} ...")

tokenizer = EsmTokenizer.from_pretrained(MODEL_NAME)
model = EsmModel.from_pretrained(MODEL_NAME)

model.to(DEVICE)
model.eval()

EMBEDDING_DIM = model.config.hidden_size
print(f"Model učitan. Dimenzija embeddinga: {EMBEDDING_DIM}")

Učitavanje modela: facebook/esm2_t6_8M_UR50D ...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model učitan. Dimenzija embeddinga: 320


### Batch ekstrakcija embeddinga (mean pooling)

Funkcija za listu proteinskih sekvenci računa per-protein embedding kao mean pooling (prosjek) reprezentacija svih aminokiselina u proteinskoj sekvenci (residue-level) iz posljednjeg skrivenog sloja ESM-2 modela, isključujući padding i specijalne tokene (<cls>, <eos>) iz proračuna prosjeka.

In [6]:
def get_batch_embeddings(sequences, tokenizer, model, device, max_length):
    inputs = tokenizer(
        sequences, 
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # (batch_size, seq_len, hidden_dim)
    last_hidden = outputs.last_hidden_state

    # attention_mask iskljucuje padding otkene (<cls>/<eos>)
    # tako sto ih postavljamo na 0 u kopiji maske koristenoj za pooling
    attention_mask = inputs["attention_mask"].clone()

    # pozicija <cls> tokena je uvijek indeks 0
    attention_mask[:, 0] = 0

    # pozicija <eos> tokena je posljednji ne-padding token u svakom redu
    seq_lengths = inputs["attention_mask"].sum(dim=1)
    for row_idx, eos_pos in enumerate(seq_lengths):
        attention_mask[row_idx, eos_pos-1] = 0

    mask_expanded = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden * mask_expanded).sum(dim=1)
    counts = mask_expanded.sum(dim=1).clamp(min=1e-9)
    mean_pooled = summed / counts

    return mean_pooled.cpu().numpy()


### Priprema za batch obradu

Sekvence se sortiraju po dužini prije batchovanja kako bi se smanjila prosječna količina paddinga unutar svakog batch-a, čime se ubrzava izvršavanje.

In [7]:
seq_df["seq_length"] = seq_df["Sequence"].str.len()
sorted_df = seq_df.sort_values("seq_length").reset_index(drop=True)

entries_sorted = sorted_df["Entry"].tolist()
sequences_sorted = sorted_df["Sequence"].tolist()

n_batches = int(np.ceil(len(sequences_sorted) / BATCH_SIZE))
print(f"Ukupan broj batch-eva: {n_batches} (batch_size={BATCH_SIZE})")

Ukupan broj batch-eva: 881 (batch_size=8)


### Glavna petlja ekstrakcije embeddinga

In [11]:
checkpoint_path = os.path.join(FEATURES_DIR, "esm2_embeddings_checkpoint.npz")

embeddings_list = []
start_batch = 0

# Ukoliko postoji prethodni checkpoint (npr. izvrsavanje je prekinuto), nastavljamo od njega
if os.path.exists(checkpoint_path):
    with np.load(checkpoint_path, allow_pickle=True) as checkpoint:
        embeddings_list = list(checkpoint["embeddings"])
        start_batch = int(checkpoint["last_batch"]) + 1

    print(f"Pronađen checkpoint - nastavljamo od batch-a {start_batch}/{n_batches}")

start_time = time.time()

for batch_idx in tqdm(range(start_batch, n_batches), initial=start_batch, total=n_batches):
    batch_start = batch_idx * BATCH_SIZE
    batch_end = min(batch_start + BATCH_SIZE, len(sequences_sorted))
    batch_sequences = sequences_sorted[batch_start:batch_end]

    batch_embeddings = get_batch_embeddings(
        batch_sequences, tokenizer, model, DEVICE, MAX_LENGTH
    )
    embeddings_list.extend(batch_embeddings)

    if (batch_idx + 1) % CHECKPOINT_EVERY == 0 or batch_idx == n_batches -1:
        np.savez(
            checkpoint_path, 
            embeddings=np.array(embeddings_list, dtype=object),
            last_batch=batch_idx,
        )


elapsed = time.time() - start_time
print(f"Ekstrakcija završena za {elapsed / 60:.1f} minuta.")

Pronađen checkpoint - nastavljamo od batch-a 881/881


100%|##########| 881/881 [00:00<?, ?it/s]

Ekstrakcija završena za 0.0 minuta.


### Sklapanje finalne matrice embeddinga 

In [12]:
embeddings_matrix_sorted = np.vstack(embeddings_list).astype(np.float32)

embeddings_df_sorted = pd.DataFrame(
    embeddings_matrix_sorted,
    index=entries_sorted,
    columns=[f"esm2_dim_{i}" for i in range(EMBEDDING_DIM)],
)

embeddings_df_sorted.index.name = "Entry"

# Vracanje u redoslijed dataset_final.cvs (Entry anchored poravnanje)
embeddings_df = embeddings_df_sorted.loc[seq_df["Entry"].values]
print(f"Dimenzije finalne matrice embeddinga: {embeddings_df.shape}")
assert embeddings_df.shape[0] == len(seq_df), "Broj redova ne odgovara broju proteina!"
assert not embeddings_df.isna().any().any(), "Pronađene NaN vrijednosti u embeddinzima!"

Dimenzije finalne matrice embeddinga: (7043, 320)


### Čuvanje embeddinga

In [13]:
embeddings_output_path = os.path.join(FEATURES_DIR, "esm2_embeddings.csv")
embeddings_df.to_csv(embeddings_output_path)

print(f"Embeddinzi sačuvani u: {embeddings_output_path}")

# Ciscenje privremenog checkpointa nakon uspjesnog cuvanja finalnog rezultata
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print("Privremeni checkpoint obrisan.")

# Cuvanje metapodataka o embeddinzima (naziv modela, dimenzija) 
metadata = {
    "model_name": MODEL_NAME,
    "embedding_dim": EMBEDDING_DIM,
    "max_length": MAX_LENGTH,
    "pooling": "mean (excluding <cls>, <eos>, padding)",
    "n_proteins": len(embeddings_df),
}
metadata_df = pd.DataFrame([metadata])
metadata_df.to_csv(os.path.join(FEATURES_DIR, "esm2_embeddings_metadata.csv"), index=False)

print("\nMetapodaci embeddinga:")
print(metadata_df.to_string(index=False))

Embeddinzi sačuvani u: ../data/features\esm2_embeddings.csv
Privremeni checkpoint obrisan.

Metapodaci embeddinga:
               model_name  embedding_dim  max_length                                pooling  n_proteins
facebook/esm2_t6_8M_UR50D            320        1022 mean (excluding <cls>, <eos>, padding)        7043


### Provjera i validacija sačuvanih embeddinga

In [14]:
loaded_embeddings = pd.read_csv(embeddings_output_path, index_col="Entry")

print(f"Učitana matrica embeddinga: {loaded_embeddings.shape}")
print(f"Broj jedinstvenih Entry vrijednosti: {loaded_embeddings.index.nunique()}")

# Provjera da li se svi Entry-jevi iz dataset_final.csv nalaze u embeddinzima
missing_entries = set(dataset["Entry"]) - set(loaded_embeddings.index)
print(f"Broj proteina iz dataset_multilabel.csv bez embeddinga: {len(missing_entries)}")

# Osnovna statistika normi vektora embeddinga (provjera da nema degenerisanih/nultih vektora)
embedding_norms = np.linalg.norm(loaded_embeddings.values, axis=1)
print(f"\nStatistika L2 normi embeddinga:")
print(f"  min:     {embedding_norms.min():.4f}")
print(f"  max:     {embedding_norms.max():.4f}")
print(f"  prosjek: {embedding_norms.mean():.4f}")
print(f"  std:     {embedding_norms.std():.4f}")

n_near_zero = (embedding_norms < 1e-6).sum()
print(f"\nBroj embeddinga sa normom bliskom nuli: {n_near_zero}")

Učitana matrica embeddinga: (7043, 320)
Broj jedinstvenih Entry vrijednosti: 7043
Broj proteina iz dataset_multilabel.csv bez embeddinga: 0

Statistika L2 normi embeddinga:
  min:     3.5374
  max:     6.7188
  prosjek: 5.0163
  std:     0.4865

Broj embeddinga sa normom bliskom nuli: 0
